# Dexm Scene Simulation & Visualization

This notebook initializes the Dexm scene and solvers via `DexmBuilder`, runs the coupled physics simulation for 3 seconds, and visualizes the robot arms and cable in Viser.

In [1]:
import sys
from pathlib import Path

# Ensure dexm package can be imported
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

import newton
import numpy as np
import warp as wp
from dexm.builder import DexmBuilder

newton.use_coord_layout_targets = True

In [2]:
def make_viewer(name: str = "dexm_scene"):
    recording_path = Path("../_static/recordings") / f"{name}.viser"
    recording_path.parent.mkdir(parents=True, exist_ok=True)
    return newton.viewer.ViewerViser(verbose=False, record_to_viser=str(recording_path))

## 1. Build Scene & Initialize Solvers
`DexmBuilder` sets up the MuJoCo + VBD coupled solver, forward kinematics, and collision pipelines.

In [3]:
dexm = DexmBuilder(worlds_count=1, fps=30, sim_substeps=10)
model = dexm.model

print(
    f"Bodies: {model.body_count}, Joints: {model.joint_count}, Shapes: {model.shape_count}"
)
print(
    f"Franka 1 bodies: {len(dexm.franka1_bodies)}, Franka 2 bodies: {len(dexm.franka2_bodies)}"
)
print(f"Cable bodies: {len(dexm.cable_bodies)}")

Warp 1.16.0 initialized:
   CUDA not enabled in this build
   Devices:
     "cpu"      : "arm"
   Kernel cache:
     /Users/runzhang/Library/Caches/warp/1.16.0
Module newton._src.geometry.inertia b20cb3b load on device 'cpu' took 0.88 ms  (cached)
Module newton._src.utils.mesh 3a50b9a load on device 'cpu' took 0.49 ms  (cached)
Module warp._src.coloring 0a2f0aa load on device 'cpu' took 1.56 ms  (cached)
Module validate_and_correct_inertia_kernel_5e18b3cd 2f0edb1 load on device 'cpu' took 1.29 ms  (cached)
Module newton._src.geometry.bvh 38d4777 load on device 'cpu' took 0.58 ms  (cached)
Module newton._src.solvers.coupled.model_view 2b7d104 load on device 'cpu' took 0.48 ms  (cached)
Module newton._src.solvers.coupled.solver_coupled 5073e24 load on device 'cpu' took 0.42 ms  (cached)
Module newton._src.solvers.mujoco.kernels 1d09d6f load on device 'cpu' took 1.09 ms  (cached)
Module mujoco_warp._src.io adacae4 load on device 'cpu' took 0.73 ms  (cached)
Module mujoco_warp._src.smooth 

## 2. Run Simulation & Visualize in Viser (3 Seconds)

In [4]:
DURATION = 3.0  # seconds
FPS = dexm.fps
SIM_SUBSTEPS = dexm.sim_substeps
SIM_DT = (1.0 / FPS) / SIM_SUBSTEPS
num_frames = int(FPS * DURATION)

# Setup Viser viewer
viewer = make_viewer("dexm_scene_sim")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.0, -1.8, 0.8), -15, 90)

sim_time = 0.0
for frame in range(num_frames):
    # Log current frame
    viewer.begin_frame(sim_time)
    viewer.log_state(dexm.state_0)
    viewer.log_contacts(dexm.contacts, dexm.state_0)
    viewer.end_frame()

    # Physics substepping
    for _ in range(SIM_SUBSTEPS):
        dexm.state_0.clear_forces()
        viewer.apply_forces(dexm.state_0)
        dexm.collision_pipeline.collide(dexm.state_0, dexm.contacts)
        dexm.solver.step(
            dexm.state_0, dexm.state_1, dexm.control, dexm.contacts, SIM_DT
        )
        newton.eval_ik(
            dexm.model, dexm.state_1, dexm.state_1.joint_q, dexm.state_1.joint_qd
        )
        dexm.state_0, dexm.state_1 = dexm.state_1, dexm.state_0

    sim_time += 1.0 / FPS

viewer

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

Module newton._src.viewer.kernels b978a3c load on device 'cpu' took 0.70 ms  (cached)
Module newton._src.sim.collide ed7af74 load on device 'cpu' took 0.56 ms  (cached)
Module newton._src.geometry.broad_phase_nxn f910460 load on device 'cpu' took 0.49 ms  (cached)
Module narrow_phase_primitive_write_contact 4440f20 load on device 'cpu' took 0.28 ms  (cached)
Module narrow_phase_gjk_mpr_True_write_contact_default_default 07afca1 load on device 'cpu' took 0.62 ms  (cached)
Module newton._src.geometry.narrow_phase ef88ac0 load on device 'cpu' took 0.44 ms  (cached)
Module newton._src.geometry.contact_reduction_global 28a460c load on device 'cpu' took 0.47 ms  (cached)
Module newton._src.geometry.sdf_contact bc58dec load on device 'cpu' took 0.40 ms  (cached)
Module narrow_phase_mesh_plane_write_contact_to_reducer_True c6c51ae load on device 'cpu' took 0.29 ms  (cached)
Module mesh_triangle_contacts_to_reducer_kernel_405db952 d6ce03f load on device 'cpu' took 0.40 ms  (cached)
Module sdf_c